set up area

In [4]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
from datetime import datetime, timedelta

load_dotenv()

API_KEY = os.getenv("API_KEY")

airports = ["ARN", "BMA", "GOT", "MMX", "LLA", "UME", "OSD", "VBY", "RNB", "KRN"]
BASE_URL = "https://api.swedavia.se/flightinfo/v2"

headers = {
    "Ocp-Apim-Subscription-Key": API_KEY,
    "Accept": "application/json"
}

today = datetime.today()
dates = [(today - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(3)]

In [7]:
print(dates[1])

2026-10-14


function

In [8]:
def fetch_flight_data(flight_type):

    df_list = []

    for airport in airports:
        for date in dates:
            url = f"{BASE_URL}/{airport}/{flight_type}/{date}"

            try:
                response = requests.get(url, headers=headers)
                response.raise_for_status()
                data = response.json()

                flights = data.get("flights", [])
                if flights:
                    df_day = pd.json_normalize(flights)
                    df_day["airportIATA"] = airport
                    df_day["queryDate"] = date
                    df_list.append(df_day)

                print(f"{airport} {date}: {len(flights)} flights")

            except Exception as e:
                print(f"{airport} {date}: {e}")

    df_all = pd.concat(df_list, ignore_index=True)
    print(f"\nFinal shape: {df_all.shape}")

    return df_all

departure to csv

In [11]:
df_departure = fetch_flight_data("departures")

df_departure.to_csv(
    "departure.csv",
    index=False
)

ARN 2026-10-15: 348 flights
ARN 2026-10-14: 342 flights
ARN 2026-10-13: 343 flights
BMA 2026-10-15: 22 flights
BMA 2026-10-14: 23 flights
BMA 2026-10-13: 23 flights
GOT 2026-10-15: 79 flights
GOT 2026-10-14: 75 flights
GOT 2026-10-13: 73 flights
MMX 2026-10-15: 22 flights
MMX 2026-10-14: 21 flights
MMX 2026-10-13: 20 flights
LLA 2026-10-15: 14 flights
LLA 2026-10-14: 16 flights
LLA 2026-10-13: 13 flights
UME 2026-10-15: 14 flights
UME 2026-10-14: 16 flights
UME 2026-10-13: 14 flights
OSD 2026-10-15: 6 flights
OSD 2026-10-14: 7 flights
OSD 2026-10-13: 6 flights
VBY 2026-10-15: 13 flights
VBY 2026-10-14: 13 flights
VBY 2026-10-13: 12 flights
RNB 2026-10-15: 6 flights
RNB 2026-10-14: 6 flights
RNB 2026-10-13: 6 flights
KRN 2026-10-15: 2 flights
KRN 2026-10-14: 2 flights
KRN 2026-10-13: 2 flights

Final shape: (1559, 40)
